## RAG - Routing
In the previous several Notebooks, we have discussed the process of Query Translation, the goal of which is to take input user question and translate it in such a way as to improve retrieval. We saw various techniques such as _Query decomposition_ (which includes techniques like Multi-Query, RAG Fusion and Step-back), and _HyDE_ (which involves generation of pseudo documents).

![RAG Pipeline](images/rag_detailed_pipeline.png)

### What is Routing?
**Routing** is the next step, which is potentially routing that decomposed query to the right source. In our RAG pipeline, we could have several sources of data, such as vector-stores, a GraphDB or an RDBMS. We simply route (or direct) the query based to the right source based upon content of the question. There are a few different ways to do that.

One of the techniques is called **Logical Routing**. In this case we basically give our LLM knowledge of the various datasources that we have at our disposal and we let the LLM _reason_ about which one to apply the question to.

![Logical Routing](images/logical_routing.png)

Alternatively, we could use **Semantic Routing**, which is where we take take a question, we embed it, embed prompts and compare the similarity betweeen our question and embedded prompts and we choose a prompt based on the similarity.

![Semantic Routing](images/semantic_routing.png)

So the general idea is to route question to different prompts (or arbitrarily taking the question and sending it to the rioht source that can answer it).

In [8]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from IPython.display import display, Markdown

from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [2]:
# load API keys from .env files
load_dotenv(override=True)
console = Console()

In [3]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_qd"

### Logical Routing

![Logical Routing](images/logical_routing.png)

In [6]:
from typing import Literal
from pydantic import BaseModel, Field


# Data model: here we define various "routes" depending on programming language
# So Python related queries should go to "python_docs", JavaScript to "js_docs" etc.
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )


structured_llm = llm.with_structured_output(RouteQuery)

In [9]:
# Prompt
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define router
router = prompt | structured_llm

In [15]:
# now let's try it out with various languages
# Python first
question_python = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})
print(result.datasource)

js_docs


In [16]:
# How about this?
question_go = """Why doesn't the following code work:

import (
	"fmt"
)

func main() {
	m := make(map[string]int)
	vals := []int{1, 2, 3}

	for _, v := range vals {
		go func() {
			m["sum"] += v
		}()
	}

	fmt.Println("sum:", m["sum"])
}
"""

result = router.invoke({"question": question})
print(result.datasource)

js_docs


In [17]:
# And this?
question_js = """Why doesn't the following code work:

let count = 0;

for (var i = 0; i < 5; i++) {
  setInterval(function () {
    count++;
    console.log("i:", i, "count:", count);
    if (count === 5) {
      clearInterval(this);
    }
  }, 1000);
}
"""

result = router.invoke({"question": question})
print(result.datasource)

js_docs


So as you can see, the LLM is able to _detect_ the programming language and _direct_ us to the correct data source to redirect our query to!

Now let us create a common function to route.

In [ ]:
def choose_route(result):
    if "python_docs" in result.datasource.lower():
        ### Logic here
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        ### Logic here
        return "chain for js_docs"
    else:
        ### Logic here
        return "golang_docs"


from langchain_core.runnables import RunnableLambda

full_chain = router | RunnableLambda(choose_route)

In [19]:
full_chain.invoke({"question": question_js})

'chain for js_docs'

In [20]:
full_chain.invoke({"question": question_python})

'chain for python_docs'

In [21]:
full_chain.invoke({"question": question_go})

'golang_docs'

### Semantic Routing

![Samantic Routing](images/semantic_routing.png)